In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
data = 'https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-03-churn-prediction/WA_Fn-UseC_-Telco-Customer-Churn.csv'
df = pd.read_csv(data)
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
df.head().T

,0,1,2,3,4
customerID,7590-VHVEG,5575-GNVDE,3668-QPYBK,7795-CFOCW,9237-HQITU
gender,Female,Male,Male,Male,Female
SeniorCitizen,0,0,0,0,0
Partner,Yes,No,No,No,No
Dependents,No,No,No,No,No
tenure,1,34,2,45,2
PhoneService,No,Yes,Yes,No,Yes
MultipleLines,No phone service,No,No,No phone service,No
InternetService,DSL,DSL,DSL,DSL,Fiber optic
OnlineSecurity,No,Yes,Yes,Yes,No


### Data Prep

In [5]:
# Replace Column Names with Lowercase and Underscores
df.columns = df.columns.str.lower().str.replace(' ', '_')

# Get categorical columns and replace spaces with underscores
cat_cols = df.select_dtypes(include=['str']).columns
df[cat_cols] = df[cat_cols].apply(lambda x: x.str.lower().str.replace(' ', '_'))

df.head()




,customerid,gender,seniorcitizen,partner,dependents,tenure,phoneservice,multiplelines,internetservice,onlinesecurity,...,deviceprotection,techsupport,streamingtv,streamingmovies,contract,paperlessbilling,paymentmethod,monthlycharges,totalcharges,churn
0,7590-vhveg,female,0,yes,no,1,no,no_phone_service,dsl,no,...,no,no,no,no,month-to-month,yes,electronic_check,29.85,29.85,no
1,5575-gnvde,male,0,no,no,34,yes,no,dsl,yes,...,yes,no,no,no,one_year,no,mailed_check,56.95,1889.5,no
2,3668-qpybk,male,0,no,no,2,yes,no,dsl,yes,...,no,no,no,no,month-to-month,yes,mailed_check,53.85,108.15,yes
3,7795-cfocw,male,0,no,no,45,no,no_phone_service,dsl,yes,...,yes,yes,no,no,one_year,no,bank_transfer_(automatic),42.30,1840.75,no
4,9237-hqitu,female,0,no,no,2,yes,no,fiber_optic,no,...,no,no,no,no,month-to-month,yes,electronic_check,70.70,151.65,yes


In [6]:
# pd.to_numeric(df.totalcharges)

In [7]:
# coerce totalcharges to numeric, replacing errors with NaN
df.totalcharges = pd.to_numeric(df.totalcharges, errors='coerce')

# get customer_id, total_charges where total charges is NaN
df[df.totalcharges.isna()][["customerid", "totalcharges"]]


,customerid,totalcharges
488,4472-lvygi,NaN
753,3115-czmzd,NaN
936,5709-lvoeq,NaN
1082,4367-nuyao,NaN
1340,1371-dwpaz,NaN
3331,7644-omvmy,NaN
3826,3213-vvolg,NaN
4380,2520-sgtta,NaN
5218,2923-arzlg,NaN
6670,4075-wkniu,NaN


In [8]:
df.totalcharges = df.totalcharges.fillna(0)
df[df.totalcharges.isna()][["customerid", "totalcharges"]]

,customerid,totalcharges


In [9]:
df.churn.value_counts()

churn
no     5174
yes    1869
Name: count, dtype: int64

In [10]:
df.churn = df.churn.map({"yes": 1, "no": 0})
# Alternative way to map churn to 1 and 0
# df.churn = (df.churn == 'Yes').astype(int)
df.churn.value_counts()

churn
0    5174
1    1869
Name: count, dtype: int64

### Get Train, Validation and Test dataset

In [11]:

from sklearn.model_selection import train_test_split

# Split the data into training and testing sets 80:20
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)

# Split the df_full_train into training and validation sets 75:25 so that 60:20:20 split is achieved
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

len(df_train), len(df_val), len(df_test)

(4225, 1409, 1409)

In [12]:
# Reset the index of the training, validation, and testing sets
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [13]:
# Get Target Variable
y_train = df_train["churn"].values
y_val = df_val["churn"].values
y_test = df_test["churn"].values

In [14]:
# Delete the Target Variable from the datasets
del df_train["churn"]
del df_val["churn"]
del df_test["churn"]

### EDA

In [15]:
# Missing Values
df_full_train.isnull().sum()

customerid          0
gender              0
seniorcitizen       0
partner             0
dependents          0
tenure              0
phoneservice        0
multiplelines       0
internetservice     0
onlinesecurity      0
onlinebackup        0
deviceprotection    0
techsupport         0
streamingtv         0
streamingmovies     0
contract            0
paperlessbilling    0
paymentmethod       0
monthlycharges      0
totalcharges        0
churn               0
dtype: int64

In [16]:
# Churn rate
df_full_train.churn.mean()

np.float64(0.2655307064252751)

In [17]:
# Alternative way to get the churn rate 
df_full_train.churn.value_counts(normalize=True)

# Churn rate is the proportion of customers who churned i.e., 1 in the churn column

churn
0    0.734469
1    0.265531
Name: proportion, dtype: float64

In [18]:
# Numeric columns
numerical = ['tenure', 'monthlycharges', 'totalcharges']

# categorical columns
categorical = ['gender', 'seniorcitizen', 'partner', 'dependents'
               , 'phoneservice', 'multiplelines', 'internetservice', 'onlinesecurity', 
               'onlinebackup', 'deviceprotection', 'techsupport', 'streamingtv', 'streamingmovies', 
               'contract', 'paperlessbilling', 'paymentmethod']

In [19]:
# Get Unique values in each categorical column
df_full_train[categorical].nunique()

gender              2
seniorcitizen       2
partner             2
dependents          2
phoneservice        2
multiplelines       3
internetservice     3
onlinesecurity      3
onlinebackup        3
deviceprotection    3
techsupport         3
streamingtv         3
streamingmovies     3
contract            3
paperlessbilling    2
paymentmethod       4
dtype: int64

### Feature Importance

In [20]:
# Feature importance is part of Exploratory Data Analysis (EDA). 
# We want to identify which features affect the churn rate the most.
# We can use various methods such as correlation analysis, feature importance from models, or statistical tests to determine this.
# We use the global churn rate as a baseline to compare the impact of individual features.
global_churn_rate = df_full_train['churn'].mean()
global_churn_rate

np.float64(0.2655307064252751)

In [21]:
# Now we can compare the churn rate of individual features against the global churn rate to understand their impact.
churn_male = df_full_train[df_full_train['gender'] == 'male']['churn'].mean()
churn_female = df_full_train[df_full_train['gender'] == 'female']['churn'].mean()

churn_male, churn_female, global_churn_rate

(np.float64(0.26047800484932454),
 np.float64(0.2708409173643975),
 np.float64(0.2655307064252751))

In [22]:
# We can compute the difference between global and individual feature churn rates.
churn_diff_male = churn_male - global_churn_rate
churn_diff_female = churn_female - global_churn_rate

churn_diff_male, churn_diff_female
# Both are close to zero, indicating that the churn rates for male and female customers are similar to the global churn rate.
# This suggests that gender does not have a significant impact on churn in this dataset.

(np.float64(-0.005052701575950558), np.float64(0.005310210939122428))

In [23]:
# The Difference of Global - Group Churn rate if less than 0 then more likely to churn else less likely to churn
# The same can be expressed in ratio as risk = Group Churn rate / Global Churn rate, 
# if it is less than 1 then less likely to churn else more likely to churn

# Example usage:
df_full_train.partner.value_counts()

partner
no     2904
yes    2730
Name: count, dtype: int64

In [24]:
churn_partner = df_full_train[df_full_train.partner == 'yes'].churn.mean()
churn_no_partner = df_full_train[df_full_train.partner == 'no'].churn.mean()

churn_partner, churn_no_partner

(np.float64(0.20073260073260074), np.float64(0.32644628099173556))

In [25]:
churn_partner/global_churn_rate

np.float64(0.7559675618499149)

In [26]:
churn_no_partner/global_churn_rate

np.float64(1.2294106598311754)

In [27]:
# risk_ratio = 1 means the group is as likely to churn as the global average
# risk_ratio < 1 means the group is less likely to churn than the global average
# risk_ratio > 1 means the group is more likely to churn than the global average

In [28]:
 # Doing this for all the variables
from IPython.display import display

for c in categorical:
    print(c)
    df_group = df_full_train.groupby(c).churn.agg(['mean','count'])
    df_group["diff"] = df_group['mean'] - global_churn_rate
    df_group["risk"] = df_group["mean"] / global_churn_rate
    display(df_group)
    print()
    print()

gender


,mean,count,diff,risk
gender,,,,
female,0.270841,2747,0.005310,1.019998
male,0.260478,2887,-0.005053,0.980971




seniorcitizen


,mean,count,diff,risk
seniorcitizen,,,,
0,0.237098,4728,-0.028433,0.892922
1,0.413907,906,0.148377,1.558793




partner


...

,mean,count,diff,risk
contract,,,,
month-to-month,0.426533,3083,0.161002,1.606340
one_year,0.117987,1212,-0.147544,0.444343
two_year,0.028379,1339,-0.237151,0.106878




paperlessbilling


,mean,count,diff,risk
paperlessbilling,,,,
no,0.16414,2309,-0.101390,0.618159
yes,0.33594,3325,0.070409,1.265164




paymentmethod


,mean,count,diff,risk
paymentmethod,,,,
bank_transfer_(automatic),0.174475,1238,-0.091056,0.657080
credit_card_(automatic),0.152404,1227,-0.113126,0.573961
electronic_check,0.449921,1887,0.184390,1.694420
mailed_check,0.190328,1282,-0.075203,0.716782


### Mutual Info

In [29]:
# The Churn rate and risk ratio tell us how likely a group is to churn compared to the global average, 
# but it does not tell us how important the feature is in predicting churn. 
# Mutual information tells us how much we can learn about one variable by knowing the value of another variable.
# If knowing a feature value gives us a lot of information about the target variable, then that feature is important for predicting the mutual information between them is high else low

In [30]:
from sklearn.metrics import mutual_info_score

In [31]:
mutual_info_score(df_full_train['churn'], df_full_train['contract'])

0.0966515250997

In [32]:
score = mutual_info_score(df_full_train['churn'], df_full_train['gender'])
print(f"{score:.5f}")

0.00007


In [33]:
# Applying mutual information to all categorical features
def calculate_mutual_info(series):
    return mutual_info_score(df_full_train['churn'], series)

mi = df_full_train[categorical].apply(calculate_mutual_info)
mi.sort_values(ascending=False)

contract            0.096652
onlinesecurity      0.063393
techsupport         0.060935
internetservice     0.053313
onlinebackup        0.045424
paymentmethod       0.042861
deviceprotection    0.042007
streamingtv         0.030844
streamingmovies     0.030705
paperlessbilling    0.019077
dependents          0.014129
partner             0.010227
seniorcitizen       0.010059
multiplelines       0.000654
gender              0.000069
phoneservice        0.000040
dtype: float64

### Correlation

In [34]:
# Mutual Information measures the importance of categorical features in predicting the target variable.
# For numerical features, we use the correlation coefficient to measure the strength and direction of the linear relationship between two variables.
# It is always a number between -1 and 1, where 1 means a perfect positive correlation, -1 means a perfect negative correlation, and 0 means no correlation.
# A value close to 1 means, when one variable increases, the other variable also increases. 
# A value close to -1 means, when one variable increases, the other variable decreases. 
# A value close to 0 means there is no linear relationship between the two variables.



In [35]:
df_full_train[numerical].corrwith(df_full_train['churn'])
# Tenure has a negative correlation with churn, meaning that as tenure increases, the likelihood of churn decreases.

tenure           -0.344925
monthlycharges    0.188574
totalcharges     -0.193370
dtype: float64

In [36]:
# We can double check the correlation between tenure and churn by splitting the customer into two groups: those who have churned and those who have not, and then comparing the average tenure of each group.
df_full_train.groupby('churn')['tenure'].mean()

churn
0    37.438134
1    18.362968
Name: tenure, dtype: float64

In [37]:
# Also by splitting the customers into groups
df_full_train.tenure.max()

np.int64(72)

In [38]:
# We can split the customer into three groups - two months or less, 3 to 12 months, and more than 12 months - and then compare the churn rate of each group.
df_full_train['tenure_group'] = pd.cut(df_full_train['tenure'], bins=[0, 2, 12, np.inf], labels=['0-2', '3-12', '12+'])
df_full_train.groupby('tenure_group')['churn'].mean()

# Alternatively
# df_full_train[df_full_train.tenure <= 2].churn.mean()
# df_full_train[(df_full_train.tenure > 2) & (df_full_train.tenure <= 12)].churn.mean()
# df_full_train[df_full_train.tenure > 12].churn.mean()


tenure_group
0-2     0.595385
3-12    0.406481
12+     0.172060
Name: churn, dtype: float64

### One Hot Encoding

In [39]:
# One Hot Encoding is a method of converting categorical variables into a format that can be provided to ML algorithms to do a better job in prediction.
# We can use DictVectorizer to convert categorical variables into a sparse matrix of binary values, where each column represents a unique category and each row represents a sample.
# So far, we have looked at categorical variables one at a time, now we want to put all the features together and see how they affect the churn rate. 
# We can use One Hot Encoding to convert all categorical variables into a format that can be provided to ML algorithms.
# For example, in our dataset contract has three possible values: month-to-month, one_year and two_year. One row of the dataset 
# {"contract": "two_year", "tenure": 72, "monthlycharges": 115.5}
# becomes
# contract=month-to-month  contract=one_year  contract=two_year  tenure  monthlycharges
#                    0                  0                  1      72           115.5
# Only one of the contract columns is "hot" (set to 1) - that is where the name one-hot encoding comes from.

In [40]:
# Turning rows into dictionaries 
# DictVectorizer expects a list of dictionaries as input, where each dictionary represents a sample and the keys are the feature names.
dicts = df[categorical + numerical].to_dict(orient='records')
dicts[0]
# The orient='records' parameter specifies that we want to convert the DataFrame into a list of dictionaries, 
# where each dictionary represents a row in the DataFrame. Each key in the dictionary corresponds to a column name, and the value corresponds to the value in that column for that row.

{'gender': 'female',
 'seniorcitizen': 0,
 'partner': 'yes',
 'dependents': 'no',
 'phoneservice': 'no',
 'multiplelines': 'no_phone_service',
 'internetservice': 'dsl',
 'onlinesecurity': 'no',
 'onlinebackup': 'yes',
 'deviceprotection': 'no',
 'techsupport': 'no',
 'streamingtv': 'no',
 'streamingmovies': 'no',
 'contract': 'month-to-month',
 'paperlessbilling': 'yes',
 'paymentmethod': 'electronic_check',
 'tenure': 1,
 'monthlycharges': 29.85,
 'totalcharges': 29.85}

In [41]:
from sklearn.feature_extraction import DictVectorizer
dv = DictVectorizer(sparse=False)

train_dict = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

X_train.shape

(4225, 45)

In [42]:
val_dict = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dict)

### Logistic Regression

In [43]:
# For linear Regression, we used the function g(x) = w0 + w1*x1 + w2*x2 + ... + wn*xn, 
# where g(x) is the predicted value, w0 is the intercept/bias term, and w1, w2, ..., wn are the coefficients for each feature x1, x2, ..., xn.
# The output of the model was any real number
# Here in logistic regression, we would the sigmoid function to map the output of the linear regression to a probability between 0 and 1. The sigmoid function is defined as:
# g(x) = sigmoid(w0 + w1*x1 + w2*x2 + ... + wn*xn)
# Sigmoid turns any real number into a value between 0 and 1, which can be interpreted as a probability.
# Sigmoid function is defined as:
# sigmoid(z) = 1 / (1 + exp(-z)) 


In [44]:
def linear_regression(xi):
    result = w0

    for j in range(len(xi)):
        result += w[j] * xi[j]
    return result

def logistic_regression(xi):
    linear_output = linear_regression(xi)
    return 1 / (1 + np.exp(-linear_output))

### Training the Model

In [47]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
# solver='lbfgs' is the default solver in newer version of sklearn
# for older versions, you need to specify it explicitly
model.fit(X_train, y_train)

/usr/local/python/3.14.2/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following

In [48]:
model.intercept_[0]

np.float64(-0.12875819563844587)

In [49]:
model.coef_[0].round(3)

array([ 0.55 , -0.207, -0.47 , -0.052, -0.075,  0.035, -0.114, -0.048,
        0.027, -0.154, -0.305,  0.293, -0.114,  0.002, -0.215,  0.123,
       -0.036,  0.088, -0.114, -0.101,  0.238, -0.114, -0.25 , -0.234,
        0.107, -0.104, -0.023, -0.112, -0.181,  0.255, -0.089,  0.123,
       -0.25 ,  0.176, -0.151, -0.114,  0.138, -0.066, -0.114,  0.054,
        0.202, -0.114, -0.214, -0.059,  0.   ])

In [51]:
# Making predictions 
# model.predict_proba(X_val)[:, 1] gives the probability of churn for each customer in the validation set.  
# model.predict(X_val) gives the predicted class (0 or 1) for each customer in the validation set.
y_pred = model.predict_proba(X_val)[:, 1]
churn_decision = (y_pred >= 0.5)


In [ ]:
# Validation
# To see whether these predictions are any good, 
# we compare them with the actual labels of the validation set:
(y_val == churn_decision).mean()

np.float64(0.801277501774308)

In [53]:
# Another way
df_pred = pd.DataFrame()
df_pred['probability'] = y_pred
df_pred['prediction'] = churn_decision.astype(int)
df_pred['actual'] = y_val

df_pred['correct'] = df_pred.prediction == df_pred.actual
df_pred.correct.mean()

np.float64(0.801277501774308)

In [54]:
dict(zip(dv.get_feature_names_out(), model.coef_[0].round(3)))

{'contract=month-to-month': np.float64(0.55),
 'contract=one_year': np.float64(-0.207),
 'contract=two_year': np.float64(-0.47),
 'dependents=no': np.float64(-0.052),
 'dependents=yes': np.float64(-0.075),
 'deviceprotection=no': np.float64(0.035),
 'deviceprotection=no_internet_service': np.float64(-0.114),
 'deviceprotection=yes': np.float64(-0.048),
 'gender=female': np.float64(0.027),
 'gender=male': np.float64(-0.154),
 'internetservice=dsl': np.float64(-0.305),
 'internetservice=fiber_optic': np.float64(0.293),
 'internetservice=no': np.float64(-0.114),
 'monthlycharges': np.float64(0.002),
 'multiplelines=no': np.float64(-0.215),
 'multiplelines=no_phone_service': np.float64(0.123),
 'multiplelines=yes': np.float64(-0.036),
 'onlinebackup=no': np.float64(0.088),
 'onlinebackup=no_internet_service': np.float64(-0.114),
 'onlinebackup=yes': np.float64(-0.101),
 'onlinesecurity=no': np.float64(0.238),
 'onlinesecurity=no_internet_service': np.float64(-0.114),
 'onlinesecurity=yes':

### Using the Model

In [55]:
dicts_full_train = df_full_train[categorical + numerical].to_dict(orient='records')

dv = DictVectorizer(sparse=False)
X_full_train = dv.fit_transform(dicts_full_train)

y_full_train = df_full_train.churn.values

In [56]:
dicts_full_train = df_full_train[categorical + numerical].to_dict(orient='records')

dv = DictVectorizer(sparse=False)
X_full_train = dv.fit_transform(dicts_full_train)

y_full_train = df_full_train.churn.values

In [57]:
model = LogisticRegression(solver='lbfgs')
model.fit(X_full_train, y_full_train)

/usr/local/python/3.14.2/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following

In [58]:
dicts_test = df_test[categorical + numerical].to_dict(orient='records')
X_test = dv.transform(dicts_test)

y_pred = model.predict_proba(X_test)[:, 1]
churn_decision = (y_pred >= 0.5)
(churn_decision == y_test).mean()

np.float64(0.8218594748048261)

In [59]:
customer = dicts_test[-1]
customer

{'gender': 'male',
 'seniorcitizen': 0,
 'partner': 'no',
 'dependents': 'yes',
 'phoneservice': 'no',
 'multiplelines': 'no_phone_service',
 'internetservice': 'dsl',
 'onlinesecurity': 'no',
 'onlinebackup': 'yes',
 'deviceprotection': 'no',
 'techsupport': 'no',
 'streamingtv': 'no',
 'streamingmovies': 'no',
 'contract': 'month-to-month',
 'paperlessbilling': 'yes',
 'paymentmethod': 'mailed_check',
 'tenure': 1,
 'monthlycharges': 31.35,
 'totalcharges': 31.35}

In [60]:
X_small = dv.transform([customer])
model.predict_proba(X_small)[0, 1]

np.float64(0.4591439567232641)

In [61]:
y_test[-1]

np.int64(1)